# Baligh-1.7B v0 - Full Training Pipeline on Colab (HF Hub)

This notebook runs the complete pipeline:
1. Install & Setup
2. Prepare Data (CPT + SFT)
3. CPT Training (with auto-resume from checkpoints)
4. SFT Training (with auto-resume from checkpoints)
5. Merge LoRA + Quantize
6. Push to HF Hub
7. Evaluation

In [ ]:
# @title 1. Install Dependencies
!pip install -q -r requirements/training.txt
!pip install -q -e .
!pip install -q huggingface_hub[hf_transfer] optimum accelerate
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Verify GPU
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "No GPU")

In [ ]:
# @title 2. Hugging Face Login
from huggingface_hub import login
import getpass
token = getpass.getpass("Enter HF_TOKEN (write access): ")
login(token=token)

REPO_ID = "Kandil7/Baligh-1.7B"
print(f"Target repo: {REPO_ID}")

In [ ]:
# @title 3. Clone / Setup Project
!git clone https://huggingface.co/${REPO_ID} /content/Baligh 2>/dev/null || (
    echo "Cloning from GitHub..." &&
    git clone https://github.com/Kandil7/Baligh.git /content/Baligh
)
cd /content/Baligh
!git pull origin develop 2>/dev/null || true

In [ ]:
# @title 4. Prepare Data (run once, ~20 min)
!python -m src.scripts.prepare_data --stage cpt --clean --output-dir data/train_ready
!python -m src.scripts.prepare_data --stage sft --clean --output-dir data/train_ready
!ls -la data/train_ready/

In [ ]:
# @title 5. CPT Training (auto-resumes from latest checkpoint)

# Quick pipeline test (1000 steps)
!python -m src.scripts.run_cpt \
  --data-dir /content/Baligh/data/train_ready/cpt \
  --output-dir /content/Baligh/training/cpt \
  --eval-data /content/Baligh/data/train_ready/cpt_eval \
  --config configs/cpt/cpt-stage1.yaml

# If Colab disconnects, re-run this cell — it auto-resumes from latest checkpoint!

# For full CPT training (50K steps), uncomment:
# !python -m src.scripts.run_cpt \
#   --data-dir /content/Baligh/data/train_ready/cpt \
#   --output-dir /content/Baligh/training/cpt \
#   --config configs/cpt/cpt-stage2.yaml

In [ ]:
# @title 5b. List CPT Checkpoints
from baligh.training.checkpoint import CheckpointManager
manager = CheckpointManager("/content/Baligh/training/cpt")
checkpoints = manager.list_checkpoints()
if checkpoints:
    for cp in checkpoints:
        print(f"  Step {cp['step']:>6} | Loss: {cp.get('loss', 'N/A')} | {cp['path']}")
else:
    print("No checkpoints found")

In [ ]:
# @title 6. Push CPT to HF
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="/content/Baligh/training/cpt/final",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="CPT checkpoint",
    path_in_repo="cpt"
)
print("CPT pushed to HF")

In [ ]:
# @title 7. SFT Training (auto-resumes from latest checkpoint)

# Initial SFT (5K steps)
!python -m src.scripts.run_sft \
  --data-dir /content/Baligh/data/train_ready/sft \
  --output-dir /content/Baligh/training/sft \
  --base-model /content/Baligh/training/cpt/final \
  --config configs/sft/sft-stage1.yaml

# If Colab disconnects, re-run this cell — it auto-resumes from latest checkpoint!

# For full SFT (10K steps), uncomment:
# !python -m src.scripts.run_sft \
#   --data-dir /content/Baligh/data/train_ready/sft \
#   --output-dir /content/Baligh/training/sft \
#   --base-model /content/Baligh/training/cpt/final \
#   --config configs/sft/sft-stage2.yaml

In [ ]:
# @title 7b. List SFT Checkpoints
from baligh.training.checkpoint import CheckpointManager
manager = CheckpointManager("/content/Baligh/training/sft")
checkpoints = manager.list_checkpoints()
if checkpoints:
    for cp in checkpoints:
        print(f"  Step {cp['step']:>6} | Loss: {cp.get('loss', 'N/A')} | {cp['path']}")
else:
    print("No checkpoints found")

In [ ]:
# @title 8. Merge LoRA + Quantize
!python -m src.scripts.merge_lora \
  --base-model /content/Baligh/training/cpt/final \
  --adapter-path /content/Baligh/training/sft/final \
  --output-dir /content/Baligh/release/Baligh-1.7B-v0-instruct

# Quantize to GGUF (q4_k_m)
!python -m src.scripts.quantize \
  --model-path /content/Baligh/release/Baligh-1.7B-v0-instruct \
  --output-dir /content/Baligh/release/Baligh-1.7B-v0-instruct-gguf \
  --method gguf --quantization q4_k_m

In [ ]:
# @title 9. Push All Artifacts to HF
from huggingface_hub import HfApi
api = HfApi()

# Push merged instruct model
api.upload_folder(
    folder_path="/content/Baligh/release/Baligh-1.7B-v0-instruct",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="v0-instruct merged model",
    path_in_repo="instruct"
)

# Push GGUF
api.upload_folder(
    folder_path="/content/Baligh/release/Baligh-1.7B-v0-instruct-gguf",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="GGUF q4_k_m",
    path_in_repo="gguf"
)
print("All artifacts pushed to HF")

In [ ]:
# @title 10. Run Evaluation
!python -m src.scripts.run_eval \
  --model-path /content/Baligh/release/Baligh-1.7B-v0-instruct \
  --output-dir /content/eval_results \
  --benchmarks mmlu cidar islamic \
  --max-samples 100

# Push eval results
from huggingface_hub import HfApi
api = HfApi()
api.upload_folder(
    folder_path="/content/eval_results",
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Evaluation results v0",
    path_in_repo="eval"
)
print("Evaluation complete and pushed")

# Pipeline Complete!

All artifacts are now in your HF repo:
- `Kandil7/Baligh-1.7B/cpt` - CPT checkpoint
- `Kandil7/Baligh-1.7B/sft` - SFT checkpoint
- `Kandil7/Baligh-1.7B/instruct` - Merged instruct model
- `Kandil7/Baligh-1.7B/gguf` - GGUF quantized
- `Kandil7/Baligh-1.7B/eval` - Evaluation results

## Checkpoint Tips
- If Colab disconnects, re-run the training cell — it auto-resumes from the latest checkpoint
- Use the "List Checkpoints" cells to see all saved checkpoints
- Old checkpoints are automatically cleaned up (keeps last 3)
- Each checkpoint saves metadata (step, loss, learning rate, timestamp)

## Next Steps
1. Generate model card: `python -m src.scripts.generate_model_card --output README.md --mmlu-score X --cidar-rouge X --islamic-rouge X --perplexity X`
2. Push README: `api.upload_file("README.md", repo_id=REPO_ID, path_in_repo="README.md")`
3. Create GitHub Release with tag `v0`